<a href="https://colab.research.google.com/github/dennisgathu8/36CHAMBERS/blob/main/zigzagzigla.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install spotipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.8/279.8 kB 6.7 MB/s eta 0:00:00


In [2]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth

client_id = '3ebdb463c6cb44ea89f42418d130813d'
client_secret = 'bde6cf3ee5f348b6b6611b80d82e7379'
redirect_uri = 'http://localhost:8888/callback'
username = 'dennisgathu'
scope = 'playlist-modify-public'

sp = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id=client_id,
                                               client_secret=client_secret,
                                               redirect_uri=redirect_uri,
                                               scope=scope,
                                               open_browser=False))

user_info = sp.current_user()
print(f"Logged in as: {user_info['display_name']}")

Go to the following URL: https://accounts.spotify.com/authorize?client_id=3ebdb463c6cb44ea89f42418d130813d&response_type=code&redirect_uri=http%3A%2F%2Flocalhost%3A8888%2Fcallback&scope=playlist-modify-public
Enter the URL you were redirected to: http://localhost:8888/callback?code=AQCb32P0jBZbEpXwCgzDH0ZkZGnt5UtqwRwCnwYwQHBm_wkb5CSKgzqGD1_4WZY72Xkp1h85qXZ-xaL2AC02SpOZFy24ealRtGrbdxBNxkjPyQpByGEzNTSh1rWOBsL2d_Vv0z5VKC0_0ne-IRWcgaXRTnFdC2ALoXTK5m5vUXEQODzlZBz3Got9n5cIi3I147GMV596QKVM1g
Logged in as: dennisgathu


In [56]:
def get_artist_tracks(artist_name):
  results = sp.search(q=f'artist:{artist_name}', type='artist', limit=1)
  artist_id = results['artists']['items'][0]['id']
  # get albums by the artist
  albums = []
  results = sp.artist_albums(artist_id, album_type='album,single', limit=50)
  albums.extend(results['items'])
  while results['next']:
    results = sp.next(results)
    albums.extend(results['items'])
  #get tracks from each album
  track_ids = []
  for album in albums:
    tracks = sp.album_tracks(album['id'])['items']
    for track in tracks:
      if any(artist['id'] == artist_id for artist in track['artists']):
        track_ids.append(track['id'])
  return list(set(track_ids))

artist_name = 'Bey T'
track_ids = get_artist_tracks(artist_name)
print(f"Found {len(track_ids)} unique tracks for {artist_name}")

Found 33 unique tracks for Bey T


In [57]:
def create_playlist(artist_name):
  playlist_name = f"All Songs by {artist_name}"
  playlist_description = f"A complete collection of {artist_name}'s available tracks on Spotify."

  playlist = sp.user_playlist_create(user=username, name=playlist_name, public=True, description=playlist_description)
  return playlist['id']

playlist_id = create_playlist(artist_name)
print(f"Created playlist with ID: {playlist_id}")

Created playlist with ID: 6qTySlXgUJGE9GJVhR7QzZ


In [58]:
def add_tracks_to_playlist(playlist_id, track_ids):
  for i in range(0, len(track_ids), 100):
    chunk = track_ids[i:i + 100]
    sp.playlist_add_items(playlist_id, chunk)
  print(f"Added {len(track_ids)} tracks to playlist")
add_tracks_to_playlist(playlist_id, track_ids)

Added 33 tracks to playlist


In [ ]:
artist = 'Wu-Tang Clan'
track_ids = get_artist_tracks(artist_name)
playlist_id = create_playlist(artist_name)
add_tracks_to_playlist(playlist_id, track_ids)

playlist_info = sp.playlist(playlist_id)
print(f"Playlist URL: {playlist_info['external_urls']['spotify']}")

Added 399 tracks to playlist
Playlist URL: https://open.spotify.com/playlist/0DkOTREfCHlDWDX2nxQYUi
